## 14. 🆕 Esperanza de vida cohortal (E_r) y corrección del pico de la cohorte 1956

**El problema.** Hoy $E_r$ es la esperanza de vida restante a los 65 años **de período** del año $t+65$ (ONU WPP 2024). La cohorte 1956 cumple 65 en 2021 (COVID), así que su $E_r$ cae a 16,1 años (mínimo de toda la serie) frente a 17,9 de la cohorte 1950, y eso produce el **pico de 1956** en $TBP_B$ y $TBP_A$. Además, una tabla de período congela la mortalidad de un año y **subestima la longevidad** de las cohortes que todavía van a vivir mejoras.

**La solución.** Seguir a cada cohorte por el calendario: el tramo de edad $[a, a+n)$ se recorre con la probabilidad de supervivencia del año calendario $\lfloor t + a + n/2 \rfloor$ (la misma convención de punto medio que ya usa $1-\mu$ en la sección 1). $E_r^{coh}(t)$ son los años esperados que le quedan a la cohorte a los 65 siguiendo esa trayectoria.

> ⚠️ **Aclaración algebraica (corrección de la persona):** $TBP_C=\dfrac{N(1-\mu)\bar A\tau}{J\rho}$ **no contiene $E_r$**. $E_r$ aparece en $TBP_A=\dfrac{N(1-\mu)\bar A\tau}{J\,E_r\,\rho}=\dfrac{TBP_C}{E_r}$ y en $TBP_B=\dfrac{\bar A\tau}{E_r\,\rho}$. Por eso integrar $E_r$ cohortal **cambia $TBP_A$ y $TBP_B$, y deja $TBP_C$ exactamente igual**; lo que cambia es la lectura de $TBP_C$ frente a la longevidad ($TBP_A=TBP_C/E_r$). Esta sección lo verifica con asserts (14.5 y 14.6).

Esta sección **depende solo de variables ya en memoria** del notebook principal: `d`, `s`, `mort`, `TAU`, `RHO`, `DATA` y `OUT`, y de archivos que carga con rutas candidatas (`../datos/procesados`, `datos/procesados`, `DATA`) y, si no están, la URL *raw* del repositorio. **No modifica `d`.** Escribe tablas y gráficos en `OUT/salidas/` y el CSV en `datos/procesados/`.

### 14.1 Módulo (copia de `src/tbp_mortalidad.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_mortalidad.py`) y se puede correr desde la terminal: `python src/tbp_mortalidad.py --autotest` o `--comparar`. Funciones: `cargar_tablas`, `supervivencia_cohortal`, `er_cohortal`, `er_periodo`, `er_periodo_integrado`, `grupos_extrapolados`, `comparar_con_periodo`, `recalcular_tbp`, `autotest`.

In [ ]:
# ====== MÓDULO tbp_mortalidad (misma versión que src/tbp_mortalidad.py) ======
_autotest_previo = globals().get('autotest')     # el módulo EPH (sección 12) también define autotest: se conserva
# -*- coding: utf-8 -*-
"""
tbp_mortalidad.py — Esperanza de vida restante a los 65 años POR COHORTE (E_r cohortal) para el TBP.

Proyecto TBP — Índice de Cuna Vacía (Taller de Indicadores).

QUÉ HACE
    Hoy E_r = esperanza de vida a los 65 de PERÍODO del año t+65 (ONU WPP 2024). Eso tiene dos problemas:
      (a) la cohorte 1956 cumple 65 en 2021 (COVID): E_r = 16,1 (mínimo) frente a 17,9 de la cohorte 1950;
      (b) la tabla de período congela la mortalidad de ese año y subestima la longevidad de las cohortes futuras.
    Acá se sigue a la cohorte por el calendario: el tramo de edad [a, a+n) se recorre con la probabilidad de supervivencia
    p_x del año calendario  floor(t + a + n/2)  (misma convención de punto medio que `surv65_cohorte` del notebook
    principal, que da 1−μ). E_r_cohorte(t) = años esperados de vida restantes a los 65 siguiendo esa trayectoria.

MÉTODO (grupos de 5 años + grupo abierto 100+)
    Para la cohorte t, con S(65)=1 y S(a+n) = S(a)·p_x(año del tramo):
        años vividos en el tramo = n·S(a+n) + a_x·[S(a) − S(a+n)]       (a_x de la tabla ONU, en años)
        E_r = Σ_tramos (años vividos) + S(100)·e_100+(año del tramo abierto)
    Es la misma identidad que usa la ONU para L_x y T_x en una tabla de período (L_x = n·l_{x+n} + a_x·d_x), por lo que
    con mortalidad constante en el tiempo reproduce EXACTAMENTE e_65 de período (autotest). Regla alternativa
    `regla="trapecio"` (a_x = n/2) para medir cuánto importa a_x.

SUPUESTO EXPLÍCITO (rotulado): las tablas ONU llegan a 2100. Si el recorrido de la cohorte pasa de 2100 se MANTIENE
    CONSTANTE la tabla de 2100 (sin más mejoras de mortalidad). Afecta a las últimas cohortes; ver `grupos_extrapolados`.

LÍMITES
    * Es un cálculo sobre tablas ONU proyectadas (variante media): no hay "verdad" observada para cohortes futuras.
    * La convención de punto medio asigna UN año calendario a cada grupo de 5 años: suaviza shocks de un año (COVID).
    * Argentina total/varones/mujeres; sin diferenciales por nivel socioeconómico.

Uso mínimo:
    import tbp_mortalidad as m
    T = m.cargar_tablas()                                  # busca ../datos/procesados, datos/procesados; si no, URL raw
    er = m.er_cohortal(range(1950, 2036), edad=65, sexo="Total", tablas=T)
    comp = m.comparar_con_periodo(range(1950, 2036), tablas=T)
    m.autotest()
"""
import os
import urllib.request
from dataclasses import dataclass

import numpy as np
import pandas as pd

ARCHIVO_MORT = "un_wpp2024_argentina_tabla_mortalidad_abreviada.csv"
RAW_BASE_MORT = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
RUTAS_MORT = ["../datos/procesados", "datos/procesados"]
SEXOS = ("Total", "Male", "Female")
REGLAS = ("ax", "trapecio")


# ================================================================== 1) CARGA
@dataclass
class TablaSexo:
    """Tablas de período abreviadas de un sexo: matrices año × inicio de grupo de edad."""
    px: pd.DataFrame        # prob. de sobrevivir el grupo (condicional a llegar a su inicio)
    ax: pd.DataFrame        # años promedio vividos en el grupo por quienes mueren en él
    ex: pd.DataFrame        # esperanza de vida restante al inicio del grupo (período)
    starts: np.ndarray      # inicio de cada grupo (0, 1, 5, ..., 95, 100)
    spans: np.ndarray       # ancho de cada grupo (el último, abierto, es NaN)
    y0: int
    y1: int


def buscar_archivo(nombre, rutas=None, destino="."):
    """Busca `nombre` en las rutas candidatas; si no está, lo descarga de GitHub (URL raw) a `destino`."""
    for c in list(rutas or []) + RUTAS_MORT:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(destino, exist_ok=True)
    p = os.path.join(destino, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_MORT}/{nombre}", p)
    return p


def _tabla_desde_df(sub):
    if "Variant" in sub.columns:
        sub = sub[sub["Variant"] == "Medium"]
    sub = sub.copy()
    sub["AgeGrpStart"] = sub["AgeGrpStart"].astype(int)
    px = sub.pivot_table(index="Time", columns="AgeGrpStart", values="px").sort_index()
    ax = sub.pivot_table(index="Time", columns="AgeGrpStart", values="ax").sort_index()
    ex = sub.pivot_table(index="Time", columns="AgeGrpStart", values="ex").sort_index()
    g = (sub[sub.Time == sub.Time.min()].sort_values("AgeGrpStart")[["AgeGrpStart", "AgeGrpSpan"]])
    starts = g["AgeGrpStart"].to_numpy(int)
    spans = np.where(g["AgeGrpSpan"].to_numpy(float) > 0, g["AgeGrpSpan"].to_numpy(float), np.nan)
    assert px.notna().all().all() and ax.notna().all().all() and ex.notna().all().all(), "tabla con huecos"
    assert np.isnan(spans[-1]) and not np.isnan(spans[:-1]).any(), "se espera un único grupo abierto, el último"
    return TablaSexo(px, ax, ex, starts, spans, int(px.index.min()), int(px.index.max()))


def cargar_tablas(rutas=None, archivo=ARCHIVO_MORT, df=None, destino="."):
    """Lee la tabla de mortalidad abreviada de la ONU y devuelve {'Total','Male','Female'} -> TablaSexo.

    `df` permite pasar una tabla ya cargada (con la misma estructura de columnas)."""
    if df is None:
        df = pd.read_csv(buscar_archivo(archivo, rutas, destino), low_memory=False)
    out = {}
    for sx in SEXOS:
        sub = df[df["Sex"] == sx]
        if len(sub):
            out[sx] = _tabla_desde_df(sub)
    if not out:
        raise ValueError("la tabla no tiene ninguna de las columnas Sex esperadas: " + str(SEXOS))
    return out


# ================================================================== 2) SUPERVIVENCIA Y E_r COHORTAL
def _anio_tramo(t, a, ancho, tab):
    """Año calendario (punto medio, igual que surv65_cohorte) con que la cohorte t recorre el tramo [a, a+ancho)."""
    y = int(np.floor(t + a + ancho / 2.0))
    if y < tab.y0:
        raise ValueError(f"cohorte {t}: el tramo {a}+ cae en {y}, antes de la primera tabla ({tab.y0})")
    return min(y, tab.y1), y > tab.y1          # (año de tabla usado, ¿extrapolado con la tabla 2100 constante?)


def _tab(tablas, sexo):
    tablas = tablas if tablas is not None else cargar_tablas()
    if sexo not in tablas:
        raise ValueError(f"sexo {sexo!r} no disponible: {list(tablas)}")
    return tablas[sexo]


def supervivencia_cohortal(cohortes, sexo="Total", tablas=None):
    """S(a): probabilidad de que la cohorte nacida en t llegue viva a la edad a (inicio de cada grupo de edad).

    Devuelve DataFrame (índice = cohorte, columnas = 0, 1, 5, ..., 95, 100). S(0)=1 y, para cada grupo cerrado,
    S(a+n) = S(a)·p_x[año calendario floor(t+a+n/2), grupo a]. Para años > 2100 se usa la tabla de 2100 (supuesto)."""
    tab = _tab(tablas, sexo)
    cohortes = [int(c) for c in cohortes]
    filas = []
    for t in cohortes:
        S = [1.0]
        for k in range(len(tab.starts) - 1):
            y, _ = _anio_tramo(t, tab.starts[k], tab.spans[k], tab)
            S.append(S[-1] * tab.px.at[y, tab.starts[k]])
        filas.append(S)
    return pd.DataFrame(filas, index=pd.Index(cohortes, name="cohorte"), columns=tab.starts)


def _k_edad(tab, edad):
    if int(edad) not in set(tab.starts.tolist()):
        raise ValueError(f"edad {edad} no es el inicio de un grupo de la tabla abreviada: {tab.starts.tolist()}")
    return int(np.where(tab.starts == int(edad))[0][0])


def _er_una(t, k0, tab, regla):
    S, total = 1.0, 0.0
    for k in range(k0, len(tab.starts) - 1):
        a, n = tab.starts[k], tab.spans[k]
        y, _ = _anio_tramo(t, a, n, tab)
        Snext = S * tab.px.at[y, a]
        axk = tab.ax.at[y, a] if regla == "ax" else n / 2.0
        total += n * Snext + axk * (S - Snext)          # n·l_{x+n} + a_x·d_x, con l normalizado a S(edad)=1
        S = Snext
    a_op = tab.starts[-1]                                # grupo abierto 100+
    y_ref, _ = _anio_tramo(t, a_op, 0.0, tab)
    e_ref = tab.ex.at[y_ref, a_op]
    y, _ = _anio_tramo(t, a_op, e_ref, tab)              # punto medio con la duración esperada del grupo abierto
    return total + S * tab.ex.at[y, a_op]


def er_cohortal(cohortes, edad=65, sexo="Total", tablas=None, regla="ax"):
    """Años esperados de vida restantes a la edad `edad` (65) para cada cohorte, siguiendo la trayectoria calendario.

    `regla`: "ax" (a_x de la tabla ONU; reproduce la tabla de período si la mortalidad es constante) o "trapecio"
    (a_x = n/2, para medir la sensibilidad). Devuelve Series indexada por cohorte."""
    if regla not in REGLAS:
        raise ValueError(f"regla debe ser una de {REGLAS}")
    tab = _tab(tablas, sexo)
    k0 = _k_edad(tab, edad)
    cohortes = [int(c) for c in cohortes]
    return pd.Series([_er_una(t, k0, tab, regla) for t in cohortes],
                     index=pd.Index(cohortes, name="cohorte"), name=f"er_cohortal_{sexo.lower()}")


def er_periodo(cohortes, edad=65, sexo="Total", tablas=None):
    """E_r de PERÍODO: e_x de la tabla del año t+edad (lo que usa hoy el notebook, columna un_e65)."""
    tab = _tab(tablas, sexo)
    _k_edad(tab, edad)
    vals = [tab.ex.at[int(t) + int(edad), edad] if tab.y0 <= int(t) + int(edad) <= tab.y1 else np.nan for t in cohortes]
    return pd.Series(vals, index=pd.Index([int(c) for c in cohortes], name="cohorte"), name=f"er_periodo_{sexo.lower()}")


def er_periodo_integrado(anios, edad=65, sexo="Total", tablas=None, regla="ax"):
    """Reconstruye e_edad de PERÍODO integrando p_x y a_x de UNA sola tabla (la del año dado), con la misma regla que
    `er_cohortal`. Sirve para probar la integración: con regla="ax" debe coincidir con la columna `ex` de la ONU
    (salvo el redondeo de la tabla); con "trapecio" muestra cuánto importa a_x."""
    tab = _tab(tablas, sexo)
    k0 = _k_edad(tab, edad)
    out = []
    for y in anios:
        y = int(y)
        S, total = 1.0, 0.0
        for k in range(k0, len(tab.starts) - 1):
            a, n = tab.starts[k], tab.spans[k]
            Sn = S * tab.px.at[y, a]
            total += n * Sn + (tab.ax.at[y, a] if regla == "ax" else n / 2.0) * (S - Sn)
            S = Sn
        out.append(total + S * tab.ex.at[y, tab.starts[-1]])
    return pd.Series(out, index=pd.Index([int(a) for a in anios], name="anio"), name=f"e{edad}_integrado_{regla}")


def grupos_extrapolados(cohortes, edad=65, sexo="Total", tablas=None):
    """Cuántos tramos del recorrido de la cohorte (desde `edad`, incluido el abierto) usan la tabla 2100 constante."""
    tab = _tab(tablas, sexo)
    k0 = _k_edad(tab, edad)
    out = []
    for t in cohortes:
        n_ext = 0
        for k in range(k0, len(tab.starts) - 1):
            n_ext += int(_anio_tramo(int(t), tab.starts[k], tab.spans[k], tab)[1])
        n_ext += int(_anio_tramo(int(t), tab.starts[-1], 2.3, tab)[1])
        out.append(n_ext)
    return pd.Series(out, index=pd.Index([int(c) for c in cohortes], name="cohorte"), name="grupos_extrapolados")


def comparar_con_periodo(cohortes, edad=65, sexos=SEXOS, tablas=None, regla="ax"):
    """Tabla ancha por cohorte: E_r de período y cohortal por sexo, diferencia y tramos extrapolados."""
    cohortes = [int(c) for c in cohortes]
    tablas = tablas if tablas is not None else cargar_tablas()
    out = pd.DataFrame({"cohorte": cohortes}).set_index("cohorte")
    for sx in sexos:
        suf = sx.lower()
        per = er_periodo(cohortes, edad, sx, tablas)
        coh = er_cohortal(cohortes, edad, sx, tablas, regla)
        out[f"er_periodo_{suf}"] = per
        out[f"er_cohortal_{suf}"] = coh
        out[f"dif_{suf}"] = coh - per
    out["grupos_extrapolados"] = grupos_extrapolados(cohortes, edad, sexos[0], tablas)
    return out.reset_index()


# ================================================================== 3) TBP CON E_r COHORTAL
def recalcular_tbp(d, er, A, tau, rho, nombre="cohortal"):
    """Recalcula TBP con un E_r alternativo. NOTA: TBP_C = N(1−μ)Āτ/(Jρ) NO contiene E_r y no cambia;
    cambian TBP_A = TBP_C/E_r y TBP_B = Āτ/(E_r ρ). `d` necesita cohorte, N_t, S65, J; `er` es Series por cohorte."""
    e = d["cohorte"].map(er).to_numpy(float)
    out = pd.DataFrame({"cohorte": d["cohorte"].to_numpy()})
    out["TBP_C"] = d["N_t"].to_numpy() * d["S65"].to_numpy() * A * tau / (d["J"].to_numpy() * rho)
    out[f"TBP_A_{nombre}"] = out["TBP_C"] / e
    out[f"TBP_B_{nombre}"] = A * tau / (e * rho)
    return out


# ================================================================== 4) AUTOTEST CON DATOS SINTÉTICOS
def _banner(txt):
    print("=" * 78 + "\n" + txt + "\n" + "=" * 78)


def tablas_sinteticas(mejora=0.0, y0=1950, y1=2100, A=4.3e-5, B=0.09, factor_sexo=(1.0, 1.3, 0.8)):
    """Tablas abreviadas SINTÉTICAS (Gompertz: h(x) = A·e^{Bx}·e^{−mejora·(año−y0)}); NO son Argentina.

    L_x se integra numéricamente (grilla fina), por lo que a_x sale exacto. Estructura idéntica a `cargar_tablas`."""
    starts = np.array([0, 1] + list(range(5, 100, 5)) + [100])
    spans = np.array([1.0, 4.0] + [5.0] * 19 + [np.nan])
    out = {}
    for sx, fs in zip(SEXOS, factor_sexo):
        años = np.arange(y0, y1 + 1)
        px = pd.DataFrame(index=años, columns=starts, dtype=float)
        ax = px.copy(); ex = px.copy()
        for y in años:
            Ay = A * fs * np.exp(-mejora * (y - y0))
            S = lambda x: np.exp(-(Ay / B) * (np.exp(B * np.asarray(x)) - 1.0))
            for k, (a, n) in enumerate(zip(starts, spans)):
                if np.isnan(n):
                    xs = np.linspace(a, 250, 12501)
                    L = np.trapezoid(S(xs), xs) / S(a)
                    ex.at[y, a] = L; ax.at[y, a] = L; px.at[y, a] = 0.0
                else:
                    xs = np.linspace(a, a + n, 401)
                    La = np.trapezoid(S(xs), xs)                                 # ∫ S dx sobre el tramo
                    p = S(a + n) / S(a)
                    px.at[y, a] = p
                    ax.at[y, a] = (La - n * S(a + n)) / (S(a) - S(a + n))        # años vividos por los que mueren en el tramo
            # e_x al inicio de cada grupo: ∫_x^∞ S / S(x)
            for a in starts:
                xs = np.linspace(a, 250, 12501)
                ex.at[y, a] = np.trapezoid(S(xs), xs) / S(a)
        out[sx] = TablaSexo(px, ax, ex, starts, spans, y0, y1)
    return out


def _er_exacto_continuo(t, edad, mejora, y0=1950, A=4.3e-5, B=0.09, fs=1.0):
    """E_r exacto (integral numérica) de la cohorte t con riesgo h(x, año) = A·e^{Bx}·e^{−mejora·(año−y0)}, año = t+x.
    Con tope de la mejora en 2100 (tabla constante después), igual que el supuesto del módulo."""
    xs = np.linspace(0, 250, 250001)
    año = t + xs
    f = np.exp(-mejora * (np.minimum(año, 2100.0) - y0))
    h = A * fs * np.exp(B * xs) * f
    H = np.concatenate([[0.0], np.cumsum((h[1:] + h[:-1]) / 2 * np.diff(xs))])
    S = np.exp(-H)
    i = int(round(edad / (xs[1] - xs[0])))
    return np.trapezoid(S[i:], xs[i:]) / S[i], S[i]


def autotest(verbose=True):
    """Pruebas con tablas SINTÉTICAS (rotuladas): (i) mortalidad constante => cohortal = período; (ii) integración vs
    valor exacto; (iii) mejora anual => cohortal ≥ período y cerca del exacto; (iv) tabla 2100 constante; (v) errores."""
    if verbose:
        _banner("AUTOTEST CON TABLAS SINTÉTICAS (Gompertz) — NO son datos de Argentina")
    cohs = list(range(1950, 2036))

    # (i) mortalidad constante en el tiempo: cohortal == período (exacto) para los 3 sexos
    T0 = tablas_sinteticas(mejora=0.0)
    for sx in SEXOS:
        c = er_cohortal(cohs, 65, sx, T0); p = er_periodo(cohs, 65, sx, T0)
        assert np.allclose(c.values, p.values, atol=1e-9), (sx, float((c - p).abs().max()))
    if verbose:
        print("OK (i)  mortalidad constante: E_r cohortal = E_r de período en las 86 cohortes y 3 sexos (tol 1e-9)")

    # (ii) la regla con a_x reproduce la integral exacta; el trapecio (a_x = n/2) se aparta un poco
    e_ex, _ = _er_exacto_continuo(1990, 65, 0.0)
    e_ax = er_cohortal([1990], 65, "Total", T0, "ax").iloc[0]
    e_tr = er_cohortal([1990], 65, "Total", T0, "trapecio").iloc[0]
    assert abs(e_ax - e_ex) < 1e-3, (e_ax, e_ex)
    assert abs(e_tr - e_ex) < 0.05, (e_tr, e_ex)
    if verbose:
        print(f"OK (ii) integración: exacto {e_ex:.4f} | regla a_x {e_ax:.4f} (error {e_ax - e_ex:+.5f}) | trapecio {e_tr:.4f} (error {e_tr - e_ex:+.5f}) años")

    # (iii) mejora anual de la mortalidad (1 %/año): cohortal > período y cercano a la integral exacta con tabla 2100 constante
    m = 0.01
    T1 = tablas_sinteticas(mejora=m)
    c = er_cohortal(cohs, 65, "Total", T1); p = er_periodo(cohs, 65, "Total", T1)
    assert (c >= p - 1e-6).all(), c[c < p - 1e-6]
    errs = []
    for t in (1950, 1970, 2000, 2020):
        ex_, _ = _er_exacto_continuo(t, 65, m)
        errs.append(c[t] - ex_)
    assert max(abs(e) for e in errs) < 0.15, errs
    S_ap = supervivencia_cohortal([1990, 2000], "Total", T1)[65]
    for t in (1990, 2000):
        _, s_ex = _er_exacto_continuo(t, 65, m)
        assert abs(S_ap[t] - s_ex) < 2e-3, (t, S_ap[t], s_ex)
    if verbose:
        print(f"OK (iii) mejora {m:.0%}/año: E_r cohortal ≥ período en todas las cohortes (mín. dif. {(c - p).min():+.3f}, máx. {(c - p).max():+.3f}); "
              f"error vs integral exacta: {', '.join(f'{e:+.3f}' for e in errs)} años; S65 cohortal ≈ exacta (<2e-3)")

    # (iv) supuesto explícito: pasado 2100 se mantiene la tabla 2100 => la cohorte 2035 (cumple 65 en 2100) = período 2100
    assert abs(c[2035] - p[2035]) < 1e-6
    g = grupos_extrapolados([1950, 2000, 2035], 65, "Total", T1)
    assert g[1950] == 0 and g[2000] == 1 and g[2035] == 8, g.to_dict()     # 2000: solo el abierto 100+ (año 2101); 2035: 7 tramos (65-99) + el abierto
    if verbose:
        print(f"OK (iv) tabla 2100 constante: cohorte 2035 cohortal = período; tramos extrapolados 1950/2000/2035 = {g.tolist()}")

    # (v) errores esperados + identidad TBP_A = TBP_C / E_r + TBP_C sin E_r
    for malo in (lambda: er_cohortal([2000], 66, "Total", T0), lambda: er_cohortal([2000], 65, "Otro", T0),
                 lambda: er_cohortal([2000], 65, "Total", T0, "foo")):
        try:
            malo(); raise AssertionError("debía fallar")
        except ValueError:
            pass
    dd = pd.DataFrame({"cohorte": [2000, 2001], "N_t": [7e5, 6.9e5], "S65": [0.88, 0.89], "J": [1.2e7, 1.2e7]})
    r1 = recalcular_tbp(dd, pd.Series({2000: 20.0, 2001: 21.0}), 30.0, 0.2177, 0.403)
    r2 = recalcular_tbp(dd, pd.Series({2000: 25.0, 2001: 26.0}), 30.0, 0.2177, 0.403)
    assert np.allclose(r1.TBP_C, r2.TBP_C)                                             # TBP_C no depende de E_r
    assert np.allclose(r1.TBP_A_cohortal, r1.TBP_C / np.array([20.0, 21.0]))
    assert np.allclose(r1.TBP_B_cohortal, 30 * 0.2177 / (np.array([20.0, 21.0]) * 0.403))
    if verbose:
        print("OK (v)  validación de entradas; TBP_C idéntico con distinto E_r; TBP_A = TBP_C/E_r; TBP_B = Āτ/(E_r ρ)")
        _banner("FIN AUTOTEST — estos números NO son Argentina")
    return True

autotest_mortalidad = autotest
if _autotest_previo is not None:
    autotest = _autotest_previo

### 14.2 Método: supervivencia cohortal e integración

Para la cohorte nacida en $t$, con las tablas abreviadas de período de la ONU (grupos $0, 1\!-\!4, 5\!-\!9,\dots,95\!-\!99$ y el grupo abierto $100+$):

$$S(a+n)=S(a)\cdot p_x\big[\text{año}=\lfloor t+a+n/2\rfloor,\ \text{grupo }a\big],\qquad S(0)=1.$$

Con la misma regla $S(65)$ reproduce $1-\mu$ de la columna `S65` de `d` (se verifica en 14.5). Para $E_r$ se normaliza $S(65)=1$ y se integra:

$$E_r^{coh}(t)=\sum_{a=65}^{95}\Big[n\,S(a+n)+{}_na_a\,\big(S(a)-S(a+n)\big)\Big]+S(100)\,e_{100+}\ ,$$

donde ${}_na_a$ es la columna `ax` de la ONU (años que viven, en promedio, quienes mueren dentro del tramo) y $e_{100+}$ la esperanza del grupo abierto del año calendario correspondiente.

**Por qué esta regla.** Es la misma identidad con la que la ONU construye $L_x=n\,l_{x+n}+{}_na_x\,d_x$ y $T_x$ en una tabla de período. Entonces, si la mortalidad no cambiara en el tiempo, la integración cohortal **es** la de período (prueba (i) del autotest, exacta). Con $a_x=n/2$ (trapecio) el error es de centésimas de año; la comparación se muestra abajo con datos reales y con una tabla sintética con solución exacta.

**Supuesto explícito (rotulado):** las tablas llegan a **2100**. Cuando un tramo de la trayectoria cae después de 2100 se **mantiene constante la tabla de 2100** (sin más mejoras de mortalidad). La columna `grupos_extrapolados` cuenta cuántos tramos usan ese supuesto (de 8 posibles: siete de 5 años entre 65 y 99 y el grupo abierto).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

SAL14 = os.path.join(OUT, "salidas"); os.makedirs(SAL14, exist_ok=True)
TAB14 = cargar_tablas(rutas=["../datos/procesados", "datos/procesados", DATA], destino=OUT)
print("Tablas ONU cargadas:", {k: f"{v.y0}-{v.y1}, {len(v.starts)} grupos" for k, v in TAB14.items()})

# La tabla "Total" cargada aquí debe ser la misma que `mort` del notebook principal
_m = mort.pivot_table(index="Time", columns="AgeGrpStart", values="px")
assert np.allclose(_m.values, TAB14["Total"].px.values) and list(_m.columns) == list(TAB14["Total"].starts)
print("OK: tabla Total = `mort` del notebook principal")

### 14.3 Prueba de la aproximación de integración (datos reales y sintéticos)

**(a) Datos reales:** integrar $p_x$ y $a_x$ de **una sola** tabla de período debe devolver su $e_{65}$ publicado. **(b) Datos sintéticos** (rotulados): mortalidad Gompertz con solución exacta. Las pruebas (i)–(v) completas están en `autotest()`.

In [ ]:
anios14 = range(1950, 2101)
print("(a) e65 reconstruido integrando UNA tabla de período vs `ex` publicado por la ONU (151 años)")
for sx in SEXOS:
    ref = TAB14[sx].ex[65]
    e_ax  = er_periodo_integrado(anios14, 65, sx, TAB14, "ax")
    e_tr  = er_periodo_integrado(anios14, 65, sx, TAB14, "trapecio")
    assert (e_ax - ref).abs().max() < 1e-3, sx            # sólo el redondeo de la tabla
    print(f"  {sx:6s}: regla a_x -> error máx. {np.abs(e_ax - ref).max():.5f} años | trapecio (a_x=n/2) -> error máx. {np.abs(e_tr - ref).max():.3f}, medio {np.mean(e_tr - ref):+.3f} años")

# peso del grupo abierto 100+ en E_r (cohorte 2000, Total)
_S = supervivencia_cohortal([2000], "Total", TAB14)
print(f"  Peso del grupo abierto 100+: S(100)/S(65) = {(_S[100] / _S[65]).iloc[0]:.4f} (contribuye ~{(_S[100] / _S[65]).iloc[0] * TAB14['Total'].ex.loc[2101 if 2101 <= TAB14['Total'].y1 else 2100, 100]:.3f} años de ~{er_cohortal([2000], 65, 'Total', TAB14).iloc[0]:.1f})")

print("\n(b) autotest con tablas SINTÉTICAS")
autotest_mortalidad()

### 14.4 Cálculo: E_r de período vs cohortal, por sexo

`comparar_con_periodo` devuelve, para cada cohorte, $E_r$ de período (tabla del año $t+65$, lo que usa hoy `d.E65`) y cohortal, para Total, varones y mujeres.

In [ ]:
COH14 = d["cohorte"].astype(int).tolist()
cmp14 = comparar_con_periodo(COH14, 65, tablas=TAB14).set_index("cohorte")
display(cmp14.loc[[1950, 1953, 1954, 1955, 1956, 1957, 1958, 1975, 2000, 2024, 2033, 2035],
                  ["er_periodo_total", "er_cohortal_total", "dif_total", "er_periodo_male", "er_cohortal_male",
                   "er_periodo_female", "er_cohortal_female", "grupos_extrapolados"]].round(3))

### 14.5 Validaciones (asserts)

1. **(i) Mortalidad constante ⇒ cohortal = período:** test sintético del autotest (14.3).
2. **(ii) Diferencias y pico de 1956:** media y máximo de $E_r^{coh}-E_r^{per}$ en las 86 cohortes (1950–2035); la cohorte 1956 deja de ser el mínimo y la mayor caída interanual se reduce a menos de la mitad.
3. **(iii) $S_{65}$ cohortal recalculado con la función del módulo = columna `S65` de `d`.** Además, el $E_r$ de período calculado = `d.E65`.
4. **(iv) $E_r^{coh}\ge E_r^{per}$** para cohortes con mortalidad decreciente: se verifica y se **reportan las excepciones** (tolerancia 0,001 años, por el redondeo de las tablas).

In [ ]:
dd14 = d.set_index("cohorte")
tot = cmp14["er_periodo_total"], cmp14["er_cohortal_total"]
per14, coh14 = tot

# (iii) S65 cohortal = columna S65 de d ; E_r de período = d.E65
S65_14 = supervivencia_cohortal(COH14, "Total", TAB14)[65]
assert np.allclose(S65_14.values, dd14.loc[COH14, "S65"].values, atol=1e-10), np.abs(S65_14.values - dd14.loc[COH14, "S65"].values).max()
assert np.allclose(per14.values, dd14.loc[COH14, "E65"].values, atol=0.005), np.abs(per14.values - dd14.loc[COH14, "E65"].values).max()
print(f"OK (iii) S65 cohortal = d.S65 (dif. máx. {np.abs(S65_14.values - dd14.loc[COH14, 'S65'].values).max():.1e}) | E_r período = d.E65 (dif. máx. {np.abs(per14.values - dd14.loc[COH14, 'E65'].values).max():.1e})")

# (ii) diferencias y pico de la cohorte 1956
dif14 = coh14 - per14
print(f"(ii) E_r cohortal - período (Total, 86 cohortes): media {dif14.mean():+.3f} | máx. {dif14.max():+.3f} (cohorte {int(dif14.idxmax())}) | mín. {dif14.min():+.4f} (cohorte {int(dif14.idxmin())})")
for sx in ("male", "female"):
    dx = cmp14[f"dif_{sx}"]; print(f"     {sx:6s}: media {dx.mean():+.3f} | máx. {dx.max():+.3f} (cohorte {int(dx.idxmax())})")
print(f"     sin las cohortes con tramos extrapolados (>2100): media {dif14[cmp14.grupos_extrapolados == 0].mean():+.3f} sobre {(cmp14.grupos_extrapolados == 0).sum()} cohortes")
print(f"     Período : mínimo = cohorte {int(per14.idxmin())} ({per14.min():.2f}) | E_r(1956) - E_r(1950) = {per14[1956] - per14[1950]:+.2f} | mayor caída interanual {per14.diff().min():+.2f} (cohorte {int(per14.diff().idxmin())})")
print(f"     Cohortal: mínimo = cohorte {int(coh14.idxmin())} ({coh14.min():.2f}) | E_r(1956) - E_r(1950) = {coh14[1956] - coh14[1950]:+.2f} | mayor caída interanual {coh14.diff().min():+.2f} (cohorte {int(coh14.diff().idxmin())})")
assert int(per14.idxmin()) == 1956 and int(coh14.idxmin()) != 1956
assert coh14[1956] > coh14[1950] and per14[1956] < per14[1950]
assert coh14.diff().min() > 0.5 * per14.diff().min()               # la mayor caída (negativa) se reduce a menos de la mitad
assert (coh14.loc[1954:].diff().dropna() > -1e-6).all()            # desde 1954 la serie cohortal no cae (queda constante en 2033-2035 por el supuesto 2100)
print("OK (ii) el pico de 1956 desaparece: ya no es el mínimo, y desde la cohorte 1954 E_r cohortal no cae (crece, y se aplana en 2033-2035 por el supuesto 2100)")
res = coh14.diff().loc[1950:1955]
print("     Residuo honesto: antes de 1955 queda una ondulación menor (las cohortes 1953-1954 reciben el COVID en el tramo 65-69):",
      ", ".join(f"{int(k)}: {v:+.2f}" for k, v in res.items() if k > 1950))

# (iv) cohortal >= período
exc = {}
for sx in ("total", "male", "female"):
    dx = cmp14[f"dif_{sx}"]; exc[sx] = dx[dx < -1e-3]
print("(iv) E_r cohortal >= período (tol. 0,001):", {k: (v.round(3).to_dict() if len(v) else "sin excepciones") for k, v in exc.items()})
assert len(exc["total"]) == 0
print("OK (iv) Total: cohortal >= período en las 86 cohortes. Excepciones por sexo reportadas arriba (son datos, no errores del cálculo)")

### 14.6 Recalcular $TBP_A$ y $TBP_B$ con $E_r$ cohortal ($\bar A=30$ y $14{,}2$) — $TBP_C$ no cambia

Se usa `recalcular_tbp`: `TBP_C` (no depende de $E_r$), `TBP_A = TBP_C/E_r` y `TBP_B = Āτ/(E_r ρ)`. Primero se comprueba que, con $E_r$ de período, el módulo **reproduce** las columnas `TBP_*_legal`/`TBP_*_obs` de `d` (y, por lo tanto, que sólo se cambia $E_r$).

In [ ]:
A14 = {"legal": 30.0, "obs": 14.2}
out14 = cmp14[["er_periodo_total", "er_cohortal_total"]].rename(columns={"er_periodo_total": "er_periodo", "er_cohortal_total": "er_cohortal"}).copy()
out14.insert(0, "anio_65", out14.index + 65)
for nom, A in A14.items():
    rp = recalcular_tbp(d, per14, A, TAU, RHO, "periodo").set_index("cohorte")
    rc = recalcular_tbp(d, coh14, A, TAU, RHO, "cohortal").set_index("cohorte")
    # el módulo con E_r de período reproduce d ; TBP_C no cambia
    assert np.allclose(rp["TBP_A_periodo"], dd14[f"TBP_A_{nom}"], rtol=1e-3) and np.allclose(rp["TBP_B_periodo"], dd14[f"TBP_B_{nom}"], rtol=1e-3)
    assert np.allclose(rp["TBP_C"], dd14[f"TBP_C_{nom}"]) and np.allclose(rc["TBP_C"], rp["TBP_C"], rtol=0, atol=0)
    out14[f"TBP_C_{nom}"] = rc["TBP_C"]
    out14[f"TBP_A_{nom}_periodo"] = rp["TBP_A_periodo"]; out14[f"TBP_A_{nom}_cohortal"] = rc["TBP_A_cohortal"]
    out14[f"TBP_B_{nom}_periodo"] = rp["TBP_B_periodo"]; out14[f"TBP_B_{nom}_cohortal"] = rc["TBP_B_cohortal"]
    # identidades con E_r cohortal
    assert np.allclose(rc["TBP_A_cohortal"], rc["TBP_C"] / coh14) and np.allclose(rc["TBP_A_cohortal"], dd14.loc[COH14, "N_t"] * dd14.loc[COH14, "S65"] / dd14.loc[COH14, "J"] * rc["TBP_B_cohortal"])
    print(f"Ā={A:>4}: TBP_B período {rp.TBP_B_periodo.min():.3f}-{rp.TBP_B_periodo.max():.3f} (máx. en {int(rp.TBP_B_periodo.idxmax())}) | cohortal {rc.TBP_B_cohortal.min():.3f}-{rc.TBP_B_cohortal.max():.3f} (máx. en {int(rc.TBP_B_cohortal.idxmax())}) | "
          f"TBP_A período {rp.TBP_A_periodo.min():.4f}-{rp.TBP_A_periodo.max():.4f} (máx. en {int(rp.TBP_A_periodo.idxmax())}) | cohortal {rc.TBP_A_cohortal.min():.4f}-{rc.TBP_A_cohortal.max():.4f} (máx. en {int(rc.TBP_A_cohortal.idxmax())})")
for sx in ("male", "female"):
    out14[f"er_periodo_{sx}"] = cmp14[f"er_periodo_{sx}"]; out14[f"er_cohortal_{sx}"] = cmp14[f"er_cohortal_{sx}"]
out14["grupos_extrapolados"] = cmp14["grupos_extrapolados"]
print("OK: TBP_C idéntico con E_r de período y cohortal (no depende de E_r) | TBP_A = TBP_C/E_r y TBP_A = D·TBP_B se cumplen con E_r cohortal")
bB = out14["TBP_B_legal_cohortal"]; assert int(bB.idxmax()) != 1956 and int(out14["TBP_A_legal_cohortal"].idxmax()) != 1956
print("OK: con E_r cohortal el máximo de TBP_B y TBP_A ya no está en la cohorte 1956")

### 14.7 Gráficos

(a) $E_r$ de período vs cohortal por cohorte: Total, varones y mujeres. (b) $TBP_B$ y $TBP_A$ (nivel, con $\bar A=30$ y $14{,}2$) y su **índice 1950 = 100**. $TBP_C$ no se grafica porque no cambia.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
for a_, sx, tit in zip(ax, ("total", "male", "female"), ("Total", "Varones", "Mujeres")):
    a_.plot(cmp14.index, cmp14[f"er_periodo_{sx}"], color="tab:red", label="Período (tabla del año t+65)")
    a_.plot(cmp14.index, cmp14[f"er_cohortal_{sx}"], color="tab:blue", label="Cohortal (trayectoria calendario)")
    a_.axvline(1956, color="gray", ls=":"); a_.axvline(2024.5, color="gray", ls=":")
    a_.annotate("1956 (COVID\nen t+65)", (1956, cmp14.loc[1956, f"er_periodo_{sx}"]), xytext=(1962, cmp14.loc[1956, f"er_periodo_{sx}"] - 0.2), fontsize=8, arrowprops=dict(arrowstyle="->", color="gray"))
    a_.axvspan(2000, 2035, color="orange", alpha=.07)
    a_.set_title(f"E_r a los 65 años — {tit}"); a_.set_xlabel("año de nacimiento de la cohorte"); a_.grid(alpha=.3)
ax[0].set_ylabel("años"); ax[0].legend(fontsize=8, loc="upper left")
ax[1].text(2001, ax[1].get_ylim()[0] + .3, "sombreado: tramos > 2100\ncon tabla 2100 constante (supuesto)", fontsize=7.5)
plt.tight_layout(); plt.savefig(f"{SAL14}/seccion_14_er_periodo_vs_cohortal.png", dpi=130); plt.show()

fig, ax = plt.subplots(2, 2, figsize=(14, 8.4))
for A_lab, ls_, lw in (("legal", "-", 1.8), ("obs", "--", 1.3)):
    nm = "Ā = 30" if A_lab == "legal" else "Ā = 14,2"
    ax[0, 0].plot(out14.index, out14[f"TBP_B_{A_lab}_periodo"], ls_, color="tab:red", lw=lw, label=f"período ({nm})")
    ax[0, 0].plot(out14.index, out14[f"TBP_B_{A_lab}_cohortal"], ls_, color="tab:blue", lw=lw, label=f"cohortal ({nm})")
    ax[0, 1].plot(out14.index, out14[f"TBP_A_{A_lab}_periodo"], ls_, color="tab:red", lw=lw, label=f"período ({nm})")
    ax[0, 1].plot(out14.index, out14[f"TBP_A_{A_lab}_cohortal"], ls_, color="tab:blue", lw=lw, label=f"cohortal ({nm})")
ax[0, 0].set_title("TBP_B = Ā·τ / (E_r·ρ)  (nivel)"); ax[0, 1].set_title("TBP_A = N(1−μ)·Ā·τ / (J·E_r·ρ)  (nivel)")
for nm_, col_, a_ in (("TBP_B", "TBP_B_legal", ax[1, 0]), ("TBP_A", "TBP_A_legal", ax[1, 1])):
    for ver, c_ in (("período", "tab:red"), ("cohortal", "tab:blue")):
        s_ = out14[f"{col_}_{'periodo' if ver == 'período' else 'cohortal'}"]; a_.plot(out14.index, s_ / s_.loc[1950] * 100, color=c_, label=ver)
    a_.axhline(100, color="gray", ls=":"); a_.set_title(f"{nm_}: índice 1950 = 100 (no depende de Ā)")
for a_ in ax.ravel():
    a_.axvline(1956, color="gray", ls=":"); a_.axvline(2024.5, color="gray", ls=":"); a_.grid(alpha=.3); a_.legend(fontsize=8); a_.set_xlabel("año de nacimiento de la cohorte")
fig.suptitle("TBP_C no se muestra: no contiene E_r y es idéntico (assert en 14.6)", y=1.0, fontsize=10)
plt.tight_layout(); plt.savefig(f"{SAL14}/seccion_14_tbp_periodo_vs_cohortal.png", dpi=130); plt.show()

### 14.8 Tabla: cohortes 1950, 1956, 1975, 2000, 2024 y 2035 (Ā = 30)

`Δ%` = cohortal respecto de período. `TBP_C` (años) es la misma columna en ambas versiones; $TBP_A=TBP_C/E_r$.

In [ ]:
sel14 = [1950, 1956, 1975, 2000, 2024, 2035]
t14 = out14.loc[sel14, ["er_periodo", "er_cohortal", "TBP_C_legal", "TBP_B_legal_periodo", "TBP_B_legal_cohortal", "TBP_A_legal_periodo", "TBP_A_legal_cohortal", "grupos_extrapolados"]].copy()
t14["Δ% E_r"] = (t14.er_cohortal / t14.er_periodo - 1) * 100
t14["Δ% TBP_B"] = (t14.TBP_B_legal_cohortal / t14.TBP_B_legal_periodo - 1) * 100
t14["Δ% TBP_A"] = (t14.TBP_A_legal_cohortal / t14.TBP_A_legal_periodo - 1) * 100
t14["Δ% TBP_C"] = 0.0
display(t14.round({"er_periodo": 2, "er_cohortal": 2, "TBP_C_legal": 4, "TBP_B_legal_periodo": 4, "TBP_B_legal_cohortal": 4, "TBP_A_legal_periodo": 5, "TBP_A_legal_cohortal": 5, "Δ% E_r": 1, "Δ% TBP_B": 1, "Δ% TBP_A": 1}))
print("Índice 1950 = 100 (Ā no importa):")
display(pd.DataFrame({"TBP_B período": out14.TBP_B_legal_periodo / out14.TBP_B_legal_periodo[1950] * 100, "TBP_B cohortal": out14.TBP_B_legal_cohortal / out14.TBP_B_legal_cohortal[1950] * 100,
                      "TBP_A período": out14.TBP_A_legal_periodo / out14.TBP_A_legal_periodo[1950] * 100, "TBP_A cohortal": out14.TBP_A_legal_cohortal / out14.TBP_A_legal_cohortal[1950] * 100}).loc[sel14].round(1))

### 14.9 Salida: `er_cohortal_1950_2035.csv`

Una fila por cohorte: $E_r$ de período y cohortal (Total, varones, mujeres), tramos extrapolados y $TBP_A$, $TBP_B$, $TBP_C$ recalculados para $\bar A=30$ (`legal`) y $14{,}2$ (`obs`). Se escribe en `../datos/procesados/` si existe (como la sección 10); si no, en `OUT`.

In [ ]:
cols14 = (["anio_65", "er_periodo", "er_cohortal", "er_periodo_male", "er_cohortal_male", "er_periodo_female", "er_cohortal_female", "grupos_extrapolados"]
          + [f"TBP_{k}_{n}{v}" for n in ("legal", "obs") for k, v in (("C", ""), ("A", "_periodo"), ("A", "_cohortal"), ("B", "_periodo"), ("B", "_cohortal"))])
salida14 = out14[cols14].rename(columns={"er_periodo": "er_periodo_total_ref", "er_cohortal": "er_cohortal_total"}).reset_index()
salida14.insert(2, "er_periodo", salida14.pop("er_periodo_total_ref"))
DEST14 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST14, exist_ok=True)
f14 = os.path.join(DEST14, "er_cohortal_1950_2035.csv")
salida14.round(6).to_csv(f14, index=False)
assert len(salida14) == 86 and salida14.notna().all().all()
print("Escrito:", os.path.abspath(f14), salida14.shape); print("Escrito:", SAL14, "/ seccion_14_er_periodo_vs_cohortal.png y seccion_14_tbp_periodo_vs_cohortal.png")
salida14.head(3)

### 14.10 Lectura y límites

* **Qué se puede decir:** (1) con $E_r$ cohortal el pico de la cohorte 1956 **desaparece** en $TBP_B$ y $TBP_A$ (el máximo pasa a la cohorte 1950, que es lo que corresponde a una longevidad creciente); (2) $E_r^{coh}\ge E_r^{per}$ en todas las cohortes del Total: la tabla de período **subestima** la longevidad de quienes aún van a vivir mejoras; (3) $TBP_C$ **no cambia**; lo que cambia es su lectura contra la longevidad ($TBP_A=TBP_C/E_r$).
* **Qué no se puede decir:** que $E_r^{coh}$ sea un dato observado. Para las cohortes que cumplen 65 después de ~2010 es una **proyección** de la ONU (variante media); para las posteriores a ~2000, parte de la trayectoria cae después de 2100 y se usa la **tabla 2100 constante** (supuesto explícito, sesgo conservador: no hay más mejoras). Las cohortes 2033–2035 coinciden con el período por construcción.
* **Residuo:** queda una ondulación menor en las cohortes 1953–1954 (el COVID de 2020–2021 cae en su tramo 65–69). El método asigna un solo año calendario a cada tramo de 5 años, así que **suaviza** shocks de un año en lugar de eliminarlos.
* **Excepciones a $E_r^{coh}\ge E_r^{per}$:** sólo en la apertura por sexo (mujeres de la cohorte 1950, −0,015 años; probable explicación, no verificada: sus tramos 65–69 y 70–74 caen en 2017 y 2022, y la mortalidad femenina de período de 2015 ya era baja respecto de la que la ONU proyecta para esos años). No afectan al Total.
* **Aproximación de integración:** con $a_x$ de la tabla el error de reconstruir $e_{65}$ de período es del orden de 5·10⁻⁵ años (redondeo); con trapecio, de hasta ~0,08 años. En el test sintético con mejora de 1 % anual, la regla subestima el $E_r$ exacto entre 0,02 y 0,08 años (convención de punto medio con año entero).
* **Mortalidad diferencial:** las tablas son nacionales por sexo; no hay diferenciales por ingreso o aportes, que podrían ser relevantes para quienes efectivamente se jubilan.